In [1]:
# /// script
# requires-python = ">=3.13"
# dependencies = [
#     "boto3",
#     "httpx",
#     "pystac",
#     "pystac-client",
#     "smart-open",
#     "stac-pydantic",
# ]
# ///

# Add web-map-links metadata to all relevant collections in the MAAP STAC

**Author:** Henry Rodman

**Date:** 2026-02-05

The [web-map-links STAC extension](https://github.com/stac-extensions/web-map-links?tab=readme-ov-file) defines several link rel types that indicate to client applications that they represent links to web map services. These links are automatically used by applications like STAC Browser and the Federated Collection Discovery app.

The plan is to add a render config and corresponding tilejson link to each collection that has image assets in a cloud-optimized format.

In [1]:
import json
import os
import urllib.parse

import boto3
import httpx
from pystac import Link
from pystac.extensions.render import Render, RenderExtension
from pystac_client import Client

stage = os.getenv("STAGE", "test")

if stage == "prod":
    STAC_LOADER_SNS_TOPIC_ARN = "arn:aws:sns:us-west-2:916098889494:MAAP-STAC-dev-pgSTAC-stacitemloaderTopicD9D06088-m0iaNFNtnXUP"
else:
    STAC_LOADER_SNS_TOPIC_ARN = "arn:aws:sns:us-west-2:916098889494:MAAP-STAC-test-pgSTAC-stacitemloaderTopicD9D06088-LutBraKgk6sT"

STAC_API_URL = "https://stac.maap-project.org"
TITILER_API_URL = "https://titiler-pgstac.maap-project.org"

WEB_MAP_LINKS_SCHEMA_URL = (
    "https://stac-extensions.github.io/web-map-links/v1.3.0/schema.json"
)

client = Client.open(STAC_API_URL)

# fetch all collections from the published catalog
collections = list(client.get_all_collections())

/home/henry/.cache/uv/environments-v2/juv-tmp-hvhl1ep3-5f1daf615978d49d/lib/python3.13/site-packages/pystac/collection.py:716: DeprecatedWarning: The collection 'icesat2-boreal' is deprecated.
  warnings.warn(
/home/henry/.cache/uv/environments-v2/juv-tmp-hvhl1ep3-5f1daf615978d49d/lib/python3.13/site-packages/pystac/collection.py:716: DeprecatedWarning: The collection 'icesat2-boreal-v2.1-agb' is deprecated.
  warnings.warn(
/home/henry/.cache/uv/environments-v2/juv-tmp-hvhl1ep3-5f1daf615978d49d/lib/python3.13/site-packages/pystac/collection.py:716: DeprecatedWarning: The collection 'icesat2-boreal-v2.1-ht' is deprecated.
  warnings.warn(
/home/henry/.cache/uv/environments-v2/juv-tmp-hvhl1ep3-5f1daf615978d49d/lib/python3.13/site-packages/pystac/collection.py:716: DeprecatedWarning: The collection 'icesat2-boreal-v3.0-agb' is deprecated.
  warnings.warn(
/home/henry/.cache/uv/environments-v2/juv-tmp-hvhl1ep3-5f1daf615978d49d/lib/python3.13/site-packages/pystac/collection.py:716: Depreca

Check to see what renders/tilejson/web-map-links metadata already exists:

In [2]:
viz_status = {}
for collection in collections:
    has_web_map_links = any(
        "web-map-links" in ext for ext in collection.stac_extensions
    )
    has_tilejson = any(link.rel == "tilejson" for link in collection.links)
    has_render = collection.ext.has("render")
    viz_status[collection] = {
        "renders": list(collection.ext.render.keys()) if has_render else None,
        "tilejson": has_tilejson,
        "web-map-links": (
            [link for link in collection.links if link.rel in ["xyz", "tilejson"]]
            if has_web_map_links
            else None
        ),
    }

Some collections already have web-map-links (just icesat2-boreal-v3.0+)

In [3]:
for collection, status in viz_status.items():
    if links := status.get("web-map-links"):
        print(collection.id, links)

icesat2-boreal-v3.0-agb [<Link rel=tilejson target=https://titiler.maap-project.org/collections/icesat2-boreal-v3.0-agb/WebMercatorQuad/tilejson.json?assets=cog&bidx=1&colormap_name=viridis&rescale=0,125&minzoom=6&maxzoom=18>]
icesat2-boreal-v3.0-ht [<Link rel=tilejson target=https://titiler.maap-project.org/collections/icesat2-boreal-v3.0-ht/WebMercatorQuad/tilejson.json?assets=cog&bidx=1&colormap_name=inferno&rescale=0,30&minzoom=6&maxzoom=18>]
icesat2-boreal-v3.1-agb [<Link rel=tilejson target=https://titiler.maap-project.org/collections/icesat2-boreal-v3.1-agb/WebMercatorQuad/tilejson.json?assets=cog&bidx=1&colormap_name=viridis&rescale=0,125&minzoom=6&maxzoom=18>]
icesat2-boreal-v3.1-ht [<Link rel=tilejson target=https://titiler.maap-project.org/collections/icesat2-boreal-v3.1-ht/WebMercatorQuad/tilejson.json?assets=cog&bidx=1&colormap_name=inferno&rescale=0,30&minzoom=6&maxzoom=18>]


Collections that have a render config but no web-map-links will be easy! We can get the tilejson links from the titiler API automagically, we just need to pick one of the render configs to use.

In [4]:
for collection, status in viz_status.items():
    if status["renders"] and not status["web-map-links"]:
        print(collection.id, status["renders"])

glad-glclu2020-change-v2 ['2000-2020 change']
glad-glclu2020-v2 ['2000', '2005', '2010', '2015', '2020']
glad-global-forest-change-1.11 ['gain', 'lossyear', 'treecover2000']
global-mangrove-watch-3.0 ['change', 'mangroves']
icesat2-boreal ['agb']
icesat2-boreal-v2.1-agb ['agb_viridis', 'agb_gist_earth_r']
icesat2-boreal-v2.1-ht ['ht_inferno']


In [5]:
render_choices = {
    "glad-glclu2020-change-v2": "2000-2020 change",
    "glad-glclu2020-v2": "2020",
    "glad-global-forest-change-1.11": "lossyear",
    "global-mangrove-watch-3.0": "mangroves",
    "icesat2-boreal": "agb",
    "icesat2-boreal-v2.1-agb": "agb_viridis",
    "icesat2-boreal-v2.1-ht": "ht_inferno",
}

These are collections that would benefit from new collection-level render configs (and web-map-links) so we can define the visualization parameters

In [6]:
new_render_params = {
    "ESACCI_Biomass_L4_AGB_V4_100m": (
        "biomass",
        {
            "title": "biomass",
            "assets": ["estimates"],
            "rescale": [[0, 400]],
            "colormap_name": "gist_earth_r",
            "color_formula": "gamma r 1.05",
        },
    ),
    "SRTMGL1_COD": (
        "hillshade",
        {
            "title": "hillshade",
            "assets": ["cog_default"],
            "algorithm": "hillshade",
            "buffer": 3,
            "z_exaggeration": 1e-06,
            "angle_altitude": 70,
        },
    ),
    "NCEO_Africa_AGB_100m_2017": (
        "biomass",
        {
            "title": "biomass",
            "assets": ["biomass"],
            "rescale": [[0, 400]],
            "colormap_name": "gist_earth_r",
            "color_formula": "gamma r 1.05",
        },
    ),
}

Apply the new render configs and add the tilejson links to each affected collection.

In [7]:
new_collections = {}
collections = list(client.get_all_collections())

for collection in collections:
    if render_key := render_choices.get(collection.id):
        titiler_info_url = f"{TITILER_API_URL}/collections/{collection.id}/info"
        titiler_info = httpx.get(titiler_info_url).json()
        link = next(
            (
                link
                for link in titiler_info["links"]
                if (render_key in link["title"] and link["rel"] == "tilejson")
            ),
            None,
        )
        tilejson_href = link["href"].format(tileMatrixSetId="WebMercatorQuad")
    elif collection.id in new_render_params:
        render_key, render_params = new_render_params.get(collection.id)
        tilejson_href = f"{TITILER_API_URL}/collections/{collection.id}/WebMercatorQuad/tilejson.json?{urllib.parse.urlencode(render_params, doseq=True)}"

        collection.ext.add("render")
        RenderExtension.ext(collection).apply({render_key: Render(render_params)})

    else:
        continue

    new_link = Link(
        rel="tilejson",
        target=tilejson_href,
        title=f"TileJSON link for {render_key} visualization",
        extra_fields={"render": render_key}
        if collection.id in new_render_params
        else None,
    )

    print(collection.id, new_link.to_dict())

    collection.add_link(new_link)
    collection.stac_extensions.append(WEB_MAP_LINKS_SCHEMA_URL)

    collection.validate()

    new_collections[collection.id] = collection.to_dict()

ESACCI_Biomass_L4_AGB_V4_100m {'rel': 'tilejson', 'href': 'https://titiler-pgstac.maap-project.org/collections/ESACCI_Biomass_L4_AGB_V4_100m/WebMercatorQuad/tilejson.json?title=biomass&assets=estimates&rescale=%5B0%2C+400%5D&colormap_name=gist_earth_r&color_formula=gamma+r+1.05', 'title': 'TileJSON link for biomass visualization', 'render': 'biomass'}
glad-glclu2020-change-v2 {'rel': 'tilejson', 'href': 'https://titiler-pgstac.maap-project.org/collections/glad-glclu2020-change-v2/WebMercatorQuad/tilejson.json?datetime=2000-01-01T00%3A00%3A00Z%2F2020-12-31T23%3A59%3A59Z&colormap=%7B%220%22%3A+%5B254%2C+254%2C+204%5D%2C+%221%22%3A+%5B250%2C+250%2C+195%5D%2C+%222%22%3A+%5B247%2C+247%2C+187%5D%2C+%223%22%3A+%5B244%2C+244%2C+179%5D%2C+%224%22%3A+%5B241%2C+241%2C+171%5D%2C+%225%22%3A+%5B237%2C+237%2C+162%5D%2C+%226%22%3A+%5B234%2C+234%2C+154%5D%2C+%227%22%3A+%5B231%2C+231%2C+146%5D%2C+%228%22%3A+%5B228%2C+228%2C+138%5D%2C+%229%22%3A+%5B224%2C+224%2C+129%5D%2C+%2210%22%3A+%5B221%2C+221%2C+121

Post to the STAC Loader SNS topic to ingest to pgstac

In [9]:
# post collections to StacLoader
sns_client = boto3.client("sns")

for collection_id, collection in new_collections.items():
    response = sns_client.publish(
        TopicArn=STAC_LOADER_SNS_TOPIC_ARN, Message=json.dumps(collection)
    )
    print(collection["id"], response["ResponseMetadata"]["HTTPStatusCode"])

ESACCI_Biomass_L4_AGB_V4_100m 200
glad-glclu2020-change-v2 200
glad-glclu2020-v2 200
glad-global-forest-change-1.11 200
global-mangrove-watch-3.0 200
icesat2-boreal 200
icesat2-boreal-v2.1-agb 200
icesat2-boreal-v2.1-ht 200
NCEO_Africa_AGB_100m_2017 200
SRTMGL1_COD 200
